*****************************************************************
#  The Social Web: data representation
- Instructors: Davide Ceolin, Jacco van Ossenbruggen, Filip Ilievski and Zubaria Inayat.
- TAs: Manav Neema, Mateusz Kielan, Goktu Genckaya and Chiara Michelutti.
- Exercises for Hands-on session 2 (12 November 2026)
*****************************************************************


In this session you are going to mine data in various microformats. You will see the differences in what each format can contain and what purpose it serves. We start with geographical data and then work with recipe and Schema.org metadata.

**Prerequisites:**
- Python 3.8 or newer
- Python packages: `requests`, `beautifulsoup4`, `rdflib`, `rdflib-microdata`, `cloudscraper`, `scrape-schema-recipe`

Run the installation cell once in the environment used by this notebook. The second installation cell is an alternative for systems where the `pip` command is not available.


In [ ]:
!pip install requests
!pip install beautifulsoup4
!pip install rdflib
!pip install rdflib-microdata
!pip install cloudscraper
!pip install scrape-schema-recipe

> **Info:** If you get `command not found: pip`, try the `pip3` installation cell instead.


In [ ]:
!pip3 install requests
!pip3 install beautifulsoup4
!pip3 install rdflib
!pip3 install rdflib-microdata
!pip3 install cloudscraper
!pip3 install scrape-schema-recipe


## Exercise 1

Even when a webpage does not use a microformat, useful data can often be extracted from its HTML. In this exercise you will use [Beautiful Soup][b] to find the first image in the infobox table of the [Wikipedia page on Amsterdam][a].

**Tip:** compare the code below with the page source. The image URL is in the `src` attribute of an `img` element inside the `table` with class `infobox`.

[b]: https://beautiful-soup-4.readthedocs.io/en/latest/
[a]: https://en.wikipedia.org/wiki/Amsterdam


In [ ]:
# -*- coding: utf-8 -*-

import requests
from bs4 import BeautifulSoup

URL = 'https://en.wikipedia.org/wiki/Amsterdam'
response = requests.get(URL, headers={'User-Agent': 'Social Web Course Student'}, timeout=30)
response.raise_for_status()
soup = BeautifulSoup(response.text, 'html.parser')

infobox = soup.find('table', class_='infobox')
if infobox is None:
    raise ValueError('The page does not contain an infobox.')
image = infobox.find('img')
if image is None or not image.get('src'):
    raise ValueError('The infobox does not contain an image with a src attribute.')

image_url = image['src']
if image_url.startswith('//'):
    image_url = 'https:' + image_url
print(image_url)


> 💡 **Info:** If you are on a Mac and you see an issue with the ssl version, try running `brew install openssl@1.1`.

Let's look at another example:

The code below extracts coordinates from a webpage and reformats them into the geo microformat (based on Example 8-1 in *Mining the Social Web*). Wikipedia pages may encode latitude and longitude in different ways. The Amsterdam page uses a `span` element containing both values:

```html
<span class="geo">52.367; 4.900</span>
```

The values are separated by a semicolon, so the code handles that representation as well as pages that provide separate `latitude` and `longitude` elements.


In [ ]:
geo_tag = soup.find(class_='geo')
if geo_tag is None:
    print('Location not found')
else:
    latitude_tag = geo_tag.find(class_='latitude')
    longitude_tag = geo_tag.find(class_='longitude')
    if latitude_tag is not None and longitude_tag is not None:
        lat = latitude_tag.get_text(strip=True)
        lon = longitude_tag.get_text(strip=True)
    else:
        coordinates = geo_tag.get_text(' ', strip=True).split(';')
        if len(coordinates) != 2:
            raise ValueError('The geo value is not in the expected latitude;longitude format.')
        lat, lon = (value.strip() for value in coordinates)
    print('Location is at', lat, lon)


### Task 1: From Scraped Coordinates to a World Map

Your goal in this task is to take the latitude and longitude coordinates you scraped in Exercise 1 and plot that location on a map. You will do this by first creating a KML (Keyhole Markup Language) file.

---

#### Task 1.1 Convert the output of Exercise 1 into KML

Your first step is to create a `.kml` file. This is just a plain text file, structured using XML (similar to HTML), that stores geographic data.

To complete this step, you will need to:
1.  Run your Python script from Exercise 1 to get the `latitude` and `longitude` values.
2.  Create a new, empty text file and save it as `amsterdam.kml`.
3.  Look at the documentation below to figure out the basic KML structure. You will need to create a "Placemark" for your location.

**Hints:**
* The most important tags you'll need to research are `<Placemark>`, `<Point>`, and `<coordinates>`.
* KML requires coordinates in the format `longitude,latitude,altitude`. Your Python script outputs `latitude` *first*. Keep that in mind.

**Documentation:**
* **Official KML Documentation:** https://developers.google.com/kml/documentation/?csw=1
* **Simple KML Example:** https://renenyffenegger.ch/notes/tools/Google-Earth/kml/index

---
#### Task 1.2 Visualize the point on a Map

Once you have your `amsterdam.kml` file, you need to visualize it. There are two ways to do this: a simple way to check your work, and a more advanced way that uses JavaScript.

**Option 1: The Simple Way (Used for Checking)**

The fastest way to see if your KML file works is to use a free online viewer.

- Go to a site like **https://kmzview.com/**.
- Load `amsterdam.kml` file.
- The map should fly to Amsterdam and show the placemark you defined. If it does, your KML file is correct!

**Option 2: The Right Way (Using the Google Maps API)**

This method involves building a custom HTML page that loads the Google Maps API and adds your KML file as a layer.

This is a multi-step process. You will have to figure out how to:

1.  **Host Your KML File:** Your KML file must be available on a public URL. You can use a service like **http://pastebin.com/** for this. After you paste your KML code, be sure to find and copy the **"RAW" format URL**.
2.  **Get an API Key:** You cannot use the Google Maps API without an API key. You can get one from the **[Google Cloud Console](https://developers.google.com/maps/documentation/javascript/get-api-key)**.
3.  **Write the Code:** Create an `index.html` file. You will need to write JavaScript that:
    * Loads the Google Maps JavaScript API (using your new API key).
    * Initializes a new map.
    * Creates a `google.maps.KmlLayer` object, passing in the **"RAW" URL** of your pastebin file.
    * Adds that KML layer to your map.

* **Hint:** The official Google Maps documentation for adding a **[KML Layer](https://developers.google.com/maps/documentation/javascript/kml)** will be your most important resource for this step.

In [ ]:
#Add your code for task 1.1 and 1.2 here

#### Task 1.3
Enter the link to your pin below.

**Link:** _Enter your url here_

#### Task 1.4 
Is KML a microformat, why (not)?

**Answer:** (add your answer here)

## Exercise 2 
In order to find information in the web, we can use microformats such as [hRecipe](https://microformats.org/wiki/hrecipe) or Schema.org's [Recipe](https://schema.org/Recipe). But first, we'll show you how to find arbitrary tags in a webpage.

### Task 2 
Parsing data for a <sub><sup>veggie</sup></sub> spaghetti alla carbonara recipe (from Example 2-7 in Mining the Social Web).

In [ ]:
import cloudscraper
import json
from bs4 import BeautifulSoup

# A recipe webpage from the same website used throughout this exercise.
URL = 'https://www.acouplecooks.com/spring-vegetarian-spaghetti-carbonara'

scraper = cloudscraper.create_scraper()
response = scraper.get(URL, timeout=30)
response.raise_for_status()
soup = BeautifulSoup(response.text, 'html.parser')
listchildren = list(soup.children)
print(f'The document has {len(listchildren)} top-level children.')


We can find any element in the page through **css tag selectors** (you can find them all [here](https://www.w3schools.com/cssref/css_selectors.asp))

But in short, these are:
- `"."` for classes
- `#` for ids 
- and `plain text` for the element name

You can also combine them, so for example, looking for `".class1.class2"` would select all elements displaying both classes. For a deeper overview, please check the above link (or google "html tag selectors"). 

In [ ]:
title_element = soup.select_one('title')
if title_element is None:
    raise ValueError('The page does not contain a title element.')
print(title_element)


Not so pretty.... Use the `.text` method.

In [ ]:
print(title_element.get_text(strip=True))


The website has a block of `JSON-LD` data embedded. Try to see if you can find it in the soup object. We can load the `JSON-LD` script to work with it easier.  
Let's get a list of the ingredients.

In [ ]:
def find_recipe_data(parsed_soup):
    """Return the Recipe object from JSON-LD on a recipe page."""
    scripts = parsed_soup.find_all('script', attrs={'type': 'application/ld+json'})
    for script in scripts:
        script_text = script.get_text(strip=True)
        if not script_text:
            continue
        payload = json.loads(script_text)
        graph = payload.get('@graph', []) if isinstance(payload, dict) else payload
        if isinstance(graph, dict):
            graph = [graph]
        for item in graph:
            if not isinstance(item, dict):
                continue
            item_type = item.get('@type', [])
            types = item_type if isinstance(item_type, list) else [item_type]
            if 'Recipe' in types:
                return item
    raise ValueError('The page does not contain a Recipe object in JSON-LD.')

recipe_data = find_recipe_data(soup)
for ingredient in recipe_data.get('recipeIngredient', []):
    print(ingredient)


Let's also print out the instructions.

In [ ]:
recipe_instructions = recipe_data.get('recipeInstructions', [])
for step in recipe_instructions:
    print(step.get('text', step) if isinstance(step, dict) else step)


Websites are going to be structured differently. Look at the following JSON-DL snippet.

In [ ]:
json_example = {
    "title": "The anarchist cookbook",
    "recipeInstructions": "<ol class=\"recipeSteps\"><li>Cook the linguine according to the packet instructions. </li><li>Meanwhile, carefully crack the eggs into a small bowl and beat them with a fork. Season with a little black pepper, then stir in the ricotta finely grate in most of the lemon zest. </li><li>When the pasta has 3 minutes left, add the peas. Reserve a little cooking water, then drain the linguine and peas, and return to the pan. </li><li>Stir in the egg mixture and spinach with a wooden spoon – they'll cook gently in the residual heat. Add a little pasta water to loosen, if needed. </li><li>Share between bowls and serve with a green salad.</li></ol>",
    "ingredients": ["a lot of effort", "the right mindset"]
}

recipe_instructions = json_example["recipeInstructions"]
example_soup = BeautifulSoup(recipe_instructions, 'html.parser')

To get a nice and clean list of the instructions, step by step, we can use the `.find` method to get the first `"ol"` element,  with attribute `"class.."`, and then use `.find_all` to get all list elements in there.  
Lastly, we can `strip` the list items to obtain the instructions.

In [ ]:
list_items = example_soup.find('ol', class_='recipeSteps').find_all('li')
instructions = [item.get_text(strip=True) for item in list_items]
print(instructions)

### Task 2.1
Now it's your turn. Create a function that can scrape any recipe webpage from the same website (other websites will have different class tags). 

Make sure to:

- Return itemized content (e.g. ingredients) in a list. (You may want to use a list comprehension here)
- Not all items have been cleaned of their html markdown (see variables ```ingredients``` vs. ```instructions_unparsed```). Make sure to return a list with human readable content (i.e. by using the ```.text``` attribute).


In [ ]:
def parse_website(url):
    """Return the title, ingredients, and instructions from a recipe URL."""
    scraper = cloudscraper.create_scraper()
    response = scraper.get(url, timeout=30)
    response.raise_for_status()
    parsed_soup = BeautifulSoup(response.text, 'html.parser')

    title_element = parsed_soup.select_one('title')
    if title_element is None:
        raise ValueError('The page does not contain a title element.')

    recipe_data = find_recipe_data(parsed_soup)
    ingredients = recipe_data.get('recipeIngredient', [])
    instructions = [
        step.get('text', step) if isinstance(step, dict) else step
        for step in recipe_data.get('recipeInstructions', [])
    ]
    return {
        'name': title_element.get_text(strip=True),
        'ingredients': ingredients,
        'instructions': instructions,
    }

recipe = parse_website(URL)
print(recipe)


**Implement your function in the cell below.** Use the preceding `parse_website` example as a reference, but write your own version for another recipe page on the same website. Return a dictionary containing a name, an ingredients list, and an instructions list.


In [ ]:
# Task 2.1: implement your own recipe parser here.
# It should accept a URL and return name, ingredients, and instructions.


#### How Can We Extract Information from Multiple Websites?

**The answer:** microformats.

Instead of manually extracting information from microformats like `schema.org` or `hRecipe`, you can use a package called `scrape-schema-recipe`.

Feel free to give it a try!

### Task 2.2
`hRecipe` is a microformat specifically created for recipes.

Use `scrape-schema-recipe` to compare the ingredients of at least two dessert recipes. For each recipe, record the URL and the extracted ingredients, then describe one similarity and one difference. If a site does not expose recipe metadata in a supported format, choose another public recipe page and explain your choice.


In [ ]:
import scrape_schema_recipe

# Use the package documentation to extract recipe data from two dessert URLs.
# Record the URLs, extracted ingredients, and your comparison below.


## Exercise 3

`Schema.org` is one of the most widely used annotation formats. It is a multipurpose vocabulary created by a consortium including Yahoo!, Google, and Microsoft. It can describe `entities`, `events`, `products`, recipes, and more.

Check out the vocabulary specifications at [schema.org][s].

[s]: https://schema.org/


### Task 3 - Parsing Schema.org metadata

The installation cell includes `rdflib-microdata`, which adds microdata parsing support to RDFLib. Use the RDFLib documentation and the example below to inspect triples exposed by a public Schema.org/RDF page.


In [ ]:
from rdflib import Graph

# Example public RDF resource. The HTTPS URL and correctly spelled resource are intentional.
URL = 'https://dbpedia.org/resource/Michael_Jackson'

g = Graph()
g.parse(location=URL)

for index, (subject, predicate, obj) in enumerate(g):
    print(subject, predicate, obj)
    if index == 9:
        break


In [ ]:
print(f'Graph has {len(g)} facts')


In [ ]:
print(g.serialize(format='ttl'))


### Task 3.1
Compare the [`schema.org`][s] information about a band (you can choose any band) on [`Last.fm`][l] with the `Facebook Open Graph` information about the same band on Facebook.
- What are the differences?
- Which format do you think offers better interoperability?

Refer to the relevant specifications and note that both sites may change their markup or restrict automated requests.

[l]: https://www.last.fm/
[s]: https://schema.org/


**Answer:** (add your answer here)


### Task 3.2
Explore the various microformats at https://microformats.org/ and compare the output of the exercises with the output documented there.

Think about which microformats you may want to support in your final assignment and read how to parse them.


**Answer:** (add your answer here)
